# ch04 价格弹性分析

本 Notebook 分析同一产品在不同价格水平下的销量变化，计算品类级价格弹性系数，为定价策略提供数据支撑。

## 1. 环境配置与数据加载

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname('__file__'), '..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils import (
    OUTPUT_BASE,
    CATEGORY_LIST,
    PLT_CONFIG,
)
from src.utils.data_loader import load_preprocessed
from src.utils.output_manager import ensure_dir, save_dataframe, save_figure, save_markdown

print('环境配置完成。')

In [ ]:
# 加载清洗后的数据
df = load_preprocessed('ch01')
print(f'数据形状: {df.shape}')
df.head()

## 2. 价格-销量散点图（整体）

In [ ]:
output_dir = os.path.join(OUTPUT_BASE, 'ch04_price_elasticity')
ensure_dir(output_dir)

fig, ax = plt.subplots(figsize=PLT_CONFIG['figsize'])
ax.scatter(df['Price_USD'], df['Quantity_Sold'], alpha=0.5, s=30)
ax.set_xlabel('价格 (USD)')
ax.set_ylabel('销量')
ax.set_title('价格-销量散点图')
plt.tight_layout()
save_figure(fig, 'price_quantity_scatter.png', output_dir, PLT_CONFIG['dpi'])
plt.show()

## 3. 按品类计算价格弹性

In [ ]:
elasticity_results = []
for cat in CATEGORY_LIST:
    cat_df = df[df['Category'] == cat].copy()
    if len(cat_df) < 5:
        continue
    cat_df['log_price'] = np.log(cat_df['Price_USD'].replace(0, np.nan))
    cat_df['log_qty'] = np.log(cat_df['Quantity_Sold'].replace(0, np.nan))
    cat_df = cat_df.dropna(subset=['log_price', 'log_qty'])

    if len(cat_df) > 2:
        corr = cat_df['log_price'].corr(cat_df['log_qty'])
        from numpy.polynomial.polynomial import polyfit
        try:
            b, a = polyfit(cat_df['log_price'], cat_df['log_qty'], 1)
            elasticity_results.append({
                'Category': cat,
                'Elasticity': round(b, 4),
                'Correlation': round(corr, 4),
                'N': len(cat_df),
                'Type': '弹性' if b < -1 else ('刚性' if b > -0.5 else '单位弹性')
            })
        except:
            elasticity_results.append({
                'Category': cat,
                'Elasticity': None,
                'Correlation': round(corr, 4),
                'N': len(cat_df),
                'Type': '无法计算'
            })

elasticity_df = pd.DataFrame(elasticity_results)
save_dataframe(elasticity_df, 'price_elasticity.csv', output_dir)
elasticity_df

## 4. 品类弹性系数柱状图

In [ ]:
fig, ax = plt.subplots(figsize=PLT_CONFIG['figsize'])
valid_df = elasticity_df.dropna(subset=['Elasticity'])
colors = ['#e74c3c' if e < -1 else '#f39c12' if e < -0.5 else '#27ae60' for e in valid_df['Elasticity']]
ax.barh(valid_df['Category'], valid_df['Elasticity'], color=colors)
ax.set_xlabel('价格弹性系数')
ax.set_title('各品类价格弹性系数')
ax.axvline(x=-1, color='red', linestyle='--', alpha=0.5, label='弹性阈值(-1)')
ax.legend()
plt.tight_layout()
save_figure(fig, 'category_elasticity.png', output_dir, PLT_CONFIG['dpi'])
plt.show()

## 5. 各品类价格-销量散点图

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, cat in enumerate(CATEGORY_LIST):
    cat_df = df[df['Category'] == cat]
    axes[i].scatter(cat_df['Price_USD'], cat_df['Quantity_Sold'], alpha=0.6, s=25)
    axes[i].set_title(cat)
    axes[i].set_xlabel('价格')
    axes[i].set_ylabel('销量')
plt.suptitle('各品类价格-销量关系', fontsize=14)
plt.tight_layout()
save_figure(fig, 'category_price_quantity.png', output_dir, PLT_CONFIG['dpi'])
plt.show()

## 6. 分析总结

In [ ]:
print('=' * 50)
print('价格弹性分析 - 总结')
print('=' * 50)
for _, row in elasticity_df.iterrows():
    print(f'{row["Category"]}: 弹性系数={row["Elasticity"]}, 类型={row["Type"]}')
print('=' * 50)
print('分析结果已保存至 outputs/ch04_price_elasticity/')